In [ ]:
#import packages
from sklearn.metrics import classification_report, accuracy_score
from keras.models import Sequential
from keras.layers import Activation, Dense, Dropout, LSTM, Bidirectional, GRU, SimpleRNN, TimeDistributed, Embedding
from keras.layers.convolutional import Conv1D, MaxPooling1D
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn import preprocessing
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import precision_score
from sklearn.metrics import accuracy_score
import seaborn as sns
import pandas as pd
import numpy as np
from ta import add_all_ta_features # Library that does financial technical analysis 

#to plot within notebook
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

#for normalizing data
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(0, 1))

In [ ]:
# Importing the training set
df = pd.read_csv('Data/ATTIJARIWAFA-BANK.csv', index_col="Date", parse_dates=True)

# Add all technical analysis to the dataframe we've already loaded
df = add_all_ta_features(df, "Open", "High", "Low", "Close", "Volume", fillna=True)

print(df)

In [ ]:
# Visualize Close stock prices
df.plot.line(y="Close", use_index=True)

In [ ]:
#how many days data will be used to create series to train RNN
SERIES_LENGTH=20

In [ ]:
def scale_data(df):
    for column in df.columns:
        df[column] = preprocessing.scale(df[column].values)
    return df

In [ ]:
import numpy as np
def process_data(df):
    df["Label"] = df.rolling(5).apply(lambda x: x.iloc[1] > x.iloc[0])["Close"]

    #Dropping any Nan values
    df.dropna(inplace=True)
    
    sequence=[]
    # We want to scale the data except the label part since it is already 0 and 1
    temp=df.loc[:, df.columns != 'Label']
#     temp=scale_data(temp)
    temp = scaler.fit_transform(temp)
    # print(f"temp{temp[:30]}")
    for i in range (len(temp)-SERIES_LENGTH):
       sequence.append([np.array(temp[i:i+SERIES_LENGTH]),df.iloc[i+SERIES_LENGTH,-1]]) # iloc part is to take last column data i.e. labels

    np.random.shuffle(sequence)

    #Now we will count the sells and buys to balance the data
    # Algorithm : whichever count is less, we will take up the data upto that
    X=[]
    y=[]
    buy=[]
    sell=[]
    for seq ,label in sequence:
        if label == 0:
            sell.append([seq,label])
        else:
            buy.append([seq,label])
            
    # print(f"buy :{buy[:10]}")
    # print(f"sell :{sell[:10]}")
    
    buys=len(buy)
    sells=len(sell)
    print(f"original buys:{buys} original sells:{sells}")
    if(buys<sells):
        buy=buy[:buys]
        sell=sell[:buys]
    else:
        buy=buy[:sells]
        sell=sell[:sells]

    print(f"buys:{len(buy)} sells:{len(sell)}")
    # Concat the buys an sells and shuffle it out again
    sequence=buy+sell

    np.random.shuffle(sequence)


    for seq ,label in sequence:
        X.append(seq)
        y.append(label)

    return np.array(X),np.array(y)

In [ ]:
df.shape

In [ ]:
process_data(df)

In [ ]:
df['Label'].value_counts()

In [ ]:
# Plot target variable
plt.figure(figsize=(8,4))
sns.countplot('Label', data=df)
plt.title('Target Variable Count')
plt.show()

In [ ]:
training_size=0.8

In [ ]:
spilt_point=int(training_size*len(df))

In [ ]:
#splitting data for training and testing in ratio 80:20
train_df=df[:spilt_point]
test_df=df[spilt_point:]

In [ ]:
import warnings
warnings.filterwarnings("ignore")

train_x,train_y=process_data(train_df)

test_x,test_y=process_data(test_df)

In [ ]:
print('X_train :',train_x.shape)
print('y_train :',train_y.shape)
print('X_test :',test_x.shape)
print('y_test :',test_y.shape)

In [ ]:
def build_model_LSTM():

    model=Sequential()
    
#     model.add(Conv1D(512, kernel_size=4, activation='relu', input_shape=(train_x.shape[1:])))
#     model.add(MaxPooling1D(2))
    
    model.add(LSTM(256, input_shape=(train_x.shape[1:]), return_sequences=True))
    model.add(Dropout(0.2))

    model.add(LSTM(128, input_shape=(train_x.shape[1:]), return_sequences=True))
    model.add(Dropout(0.2))

    model.add(LSTM(64, input_shape=(train_x.shape[1:]), return_sequences=False))
    model.add(Dropout(0.2))

    model.add(Dense(32, kernel_initializer="uniform", activation='tanh'))
    model.add(Dense(1, kernel_initializer="uniform", activation='sigmoid'))


    model.compile(loss='binary_crossentropy',optimizer="adam",metrics=['accuracy']) 
    history=model.fit(train_x, train_y, batch_size=96, epochs=100)
    score=model.evaluate(test_x,test_y)
    lstm_pred=model.predict(test_x).squeeze() > 0.5
    
    print("Validation accuracy percentage",score[1])
    print("Validation loss percentage",score[0])
    print("Precision score", precision_score(test_y, lstm_pred, average='macro'))
    print("Accuracy score", accuracy_score(test_y, lstm_pred, normalize=True))
    
    return model, lstm_pred

In [ ]:
model_LSTM, lstm_pred = build_model_LSTM()

Epoch 82/100
40/40 [==============================] - 1s 33ms/step - loss: 0.0944 - accuracy: 0.9640
Epoch 83/100
40/40 [==============================] - 1s 33ms/step - loss: 0.0766 - accuracy: 0.9712
Epoch 84/100
40/40 [==============================] - 1s 33ms/step - loss: 0.0832 - accuracy: 0.9659
Epoch 85/100
40/40 [==============================] - 1s 33ms/step - loss: 0.0918 - accuracy: 0.9680
Epoch 86/100
40/40 [==============================] - 1s 33ms/step - loss: 0.0870 - accuracy: 0.9664
Epoch 87/100
40/40 [==============================] - 1s 33ms/step - loss: 0.1113 - accuracy: 0.9504
Epoch 88/100
40/40 [==============================] - 1s 34ms/step - loss: 0.0841 - accuracy: 0.9688
Epoch 89/100
40/40 [==============================] - 1s 35ms/step - loss: 0.0758 - accuracy: 0.9693
Epoch 90/100
40/40 [==============================] - 1s 34ms/step - loss: 0.0972 - accuracy: 0.9624
Epoch 91/100
40/40 [==============================] - 1s 35ms/step - loss: 0.0708 - accurac

In [ ]:
mat = confusion_matrix(test_y, lstm_pred)
labels = ['Legitimate', 'Fraudulent']
 
sns.heatmap(mat, square=True, annot=True, fmt='d', cbar=False, cmap='Blues',
            xticklabels=labels, yticklabels=labels)
 
plt.xlabel('Predicted label')
plt.ylabel('Actual label')

In [ ]:
print(classification_report(test_y, lstm_pred))

In [ ]:
def build_model_GRU():

    model=Sequential()
    
#     model.add(Conv1D(256, kernel_size=4, activation='relu', input_shape=(train_x.shape[1:])))
#     model.add(MaxPooling1D(2))
    
    model.add(GRU(256, input_shape=(train_x.shape[1:]), return_sequences=True))
    model.add(Dropout(0.2))

    model.add(GRU(128, input_shape=(train_x.shape[1:]), return_sequences=True))
    model.add(Dropout(0.2))

    model.add(GRU(64, input_shape=(train_x.shape[1:]), return_sequences=False))
    model.add(Dropout(0.2))

    model.add(Dense(32, kernel_initializer="uniform", activation='tanh'))
    model.add(Dense(1, kernel_initializer="uniform", activation='sigmoid'))


    model.compile(loss='binary_crossentropy',optimizer="adam",metrics=['accuracy'])
    history=model.fit(train_x, train_y, batch_size=96, epochs=100, validation_data=(test_x,test_y))
    score=model.evaluate(test_x,test_y)
    gru_pred=model.predict(test_x) > 0.5

    print("Validation accuracy percentage",score[1])
    print("Validation loss percentage",score[0])
    print("Precision score", precision_score(test_y, gru_pred, average='macro'))
    print("Accuracy score", accuracy_score(test_y, gru_pred, normalize=True))
    
    return model, gru_pred

In [ ]:
model_GRU, gru_pred = build_model_GRU()

40/40 [==============================] - 1s 37ms/step - loss: 0.1084 - accuracy: 0.9542 - val_loss: 0.3713 - val_accuracy: 0.8770
Epoch 59/100
40/40 [==============================] - 2s 38ms/step - loss: 0.0964 - accuracy: 0.9611 - val_loss: 0.1820 - val_accuracy: 0.9211
Epoch 60/100
40/40 [==============================] - 1s 37ms/step - loss: 0.1009 - accuracy: 0.9568 - val_loss: 0.4194 - val_accuracy: 0.8760
Epoch 61/100
40/40 [==============================] - 1s 36ms/step - loss: 0.0985 - accuracy: 0.9571 - val_loss: 0.3818 - val_accuracy: 0.8699
Epoch 62/100
40/40 [==============================] - 2s 41ms/step - loss: 0.0840 - accuracy: 0.9659 - val_loss: 0.3812 - val_accuracy: 0.8852
Epoch 63/100
40/40 [==============================] - 2s 45ms/step - loss: 0.1008 - accuracy: 0.9576 - val_loss: 0.3171 - val_accuracy: 0.8852
Epoch 64/100
40/40 [==============================] - 2s 38ms/step - loss: 0.0902 - accuracy: 0.9651 - val_loss: 0.3207 - val_accuracy: 0.8996
Epoch 65/100

In [ ]:
mat = confusion_matrix(test_y, gru_pred)
labels = ['Legitimate', 'Fraudulent']
 
sns.heatmap(mat, square=True, annot=True, fmt='d', cbar=False, cmap='Blues',
            xticklabels=labels, yticklabels=labels)
 
plt.xlabel('Predicted label')
plt.ylabel('Actual label')

In [ ]:
print(classification_report(test_y, gru_pred))

In [ ]:
def build_model_BILSTM():

    model=Sequential()
    
#     model.add(Conv1D(256, kernel_size=4, activation='relu', input_shape=(train_x.shape[1:])))
#     model.add(MaxPooling1D(2))
    
    model.add(Bidirectional(LSTM(256, input_shape=(train_x.shape[1:]), return_sequences=True)))
    model.add(Dropout(0.2))

    model.add(Bidirectional(LSTM(128, input_shape=(train_x.shape[1:]), return_sequences=True)))
    model.add(Dropout(0.2))

    model.add(Bidirectional(LSTM(64, input_shape=(train_x.shape[1:]), return_sequences=False)))
    model.add(Dropout(0.2))

    model.add(Dense(32, kernel_initializer="uniform", activation='tanh'))
    model.add(Dense(1, kernel_initializer="uniform", activation='sigmoid'))


    model.compile(loss='binary_crossentropy',optimizer="adam",metrics=['accuracy'])
    history=model.fit(train_x, train_y, batch_size=96, epochs=100, validation_data=(test_x,test_y))
    score=model.evaluate(test_x,test_y)
    bilstm_pred=model.predict(test_x) > 0.5
    
    print("Validation accuracy percentage",score[1])
    print("Validation loss percentage",score[0])
    print("Precision score", precision_score(test_y, bilstm_pred, average='macro'))
    print("Accuracy score", accuracy_score(test_y, bilstm_pred, normalize=True))
    
    return model, bilstm_pred

In [ ]:
model_BILSTM, bilstm_pred = build_model_BILSTM()

40/40 [==============================] - 3s 84ms/step - loss: 0.1142 - accuracy: 0.9499 - val_loss: 0.6485 - val_accuracy: 0.8402
Epoch 59/100
40/40 [==============================] - 3s 84ms/step - loss: 0.1106 - accuracy: 0.9563 - val_loss: 1.3064 - val_accuracy: 0.7398
Epoch 60/100
40/40 [==============================] - 3s 87ms/step - loss: 0.1925 - accuracy: 0.9206 - val_loss: 0.4572 - val_accuracy: 0.8504
Epoch 61/100
40/40 [==============================] - 3s 83ms/step - loss: 0.1101 - accuracy: 0.9558 - val_loss: 0.7205 - val_accuracy: 0.8227
Epoch 62/100
40/40 [==============================] - 3s 84ms/step - loss: 0.0983 - accuracy: 0.9608 - val_loss: 0.3588 - val_accuracy: 0.8781
Epoch 63/100
40/40 [==============================] - 4s 100ms/step - loss: 0.2204 - accuracy: 0.9118 - val_loss: 0.7345 - val_accuracy: 0.8166
Epoch 64/100
40/40 [==============================] - 4s 95ms/step - loss: 0.1371 - accuracy: 0.9427 - val_loss: 0.4208 - val_accuracy: 0.8514
Epoch 65/10

In [ ]:
mat = confusion_matrix(test_y, bilstm_pred)
labels = ['Legitimate', 'Fraudulent']
 
sns.heatmap(mat, square=True, annot=True, fmt='d', cbar=False, cmap='Blues',
            xticklabels=labels, yticklabels=labels)
 
plt.xlabel('Predicted label')
plt.ylabel('Actual label')

In [ ]:
print(classification_report(test_y, bilstm_pred))